In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np

In [3]:
from bracket_functions import create_initial_bracket, get_teams
from bracket_predictions import sequential_predictions, parallel_predictions
from constants import DATA_FILENAME, ESPN_FILE, ORDER_OF_REGIONS, DATA_DIRECTORY
import json
from scipy.stats import norm

In [4]:
gender = 'mens'

In [5]:
initial_bracket, data = create_initial_bracket(DATA_FILENAME.format(gender=gender), ORDER_OF_REGIONS[gender])

In [6]:
# Import 538's info
# espn = pd.read_csv(ESPN_FILE)
# espn_slice = (espn['gender'] == 'mens') & (espn['forecast_date'] == max(espn['forecast_date']))
# team_info = espn[espn_slice][['team_id', 'team_name', 'team_region', 'team_seed']]
# espn_data = espn[espn_slice]

espn_data = pd.read_excel(f'{DATA_DIRECTORY.format(gender=gender)}/results.xlsx')
espn_data.sort_values('rd7_win', ascending=False).head()

,team_id,team_name,team_seed,team_region,playin_flag,team_alive,rd1_win,rd2_win,rd3_win,rd4_win,rd5_win,rd6_win,rd7_win,win_odds,timestamp
0,41,Connecticut,1,East,0,1,1,1,1,1,1,1,0.254744,2.92551,2024-03-24 00:35:23.130
50,2509,Purdue,1,Midwest,0,1,1,1,1,1,1,1,0.121166,7.25311,2024-03-24 00:35:23.130
36,245,Texas A&M,9,South,0,1,1,1,0,0,0,0,0.000000,161.005,2024-03-24 00:35:23.130
37,275,Wisconsin,5,South,0,0,1,0,0,0,0,0,0.000000,--,2024-03-24 00:35:23.130
38,256,James Madison,12,South,0,1,1,1,0,0,0,0,0.000000,1812.8,2024-03-24 00:35:23.130


In [7]:
# Set championship result
espn_data.loc[0, "rd7_win"] = 1
espn_data.loc[50, "rd7_win"] = 0

In [8]:
data.head()

,64,team_region,Seed,team_id,AdjustEM,AdjustT,team_rating
0,Connecticut,East,1,41.0,31.95,64.5,95.584846
1,Stetson,East,16,56.0,-4.14,66.2,70.698620
2,Florida Atlantic,East,8,2226.0,16.37,68.1,84.880105
3,Northwestern,East,9,77.0,15.90,63.7,83.526257
4,San Diego State,East,5,21.0,20.01,66.0,85.622503


## TODO Create a py file for these

In [9]:
def create_specific_bracket(initial_bracket, data, espn, bracket_type_func):
    bracket = initial_bracket.copy()
    rounds = ['64', '32', '16', '8', '4', '2', '1']
    for round_ in range(6):
        gap = 2**(round_ + 1)  # how far apart two teams could be for a given round
        for top_team in range(0, len(bracket), gap):
            teams = get_teams(top_team, gap, bracket, rounds, round_, data)
            winner_num, winner_name = bracket_type_func(teams, data, espn, round_, bracket)
            bracket.loc[winner_num, rounds[round_ + 1]] = winner_name  # write in the winner
    return bracket

In [10]:
def get_team_ids(pyths, data):
#     return data.loc[data["64"].isin([i[1] for i in pyths])]["team_id"].tolist()
    return data.loc[pyths]["team_id"].tolist()


def get_winner(team_ids, data, round_):
    df = data.loc[lambda x: x["team_id"].isin(team_ids)]
    return df.loc[df[f"rd{round_ + 2}_win"].idxmax()]["team_name"]


def perfect_bracket(teams, data, espn, round_, *args):
    team_ids = get_team_ids(teams, data)
    winner_name = get_winner(team_ids, espn, round_)
#     winner_num = [i[0] for i in pyths if i[1] == winner_name][0]
    winner_num = data.loc[lambda x: x["64"] == winner_name].index
    
    return winner_num, winner_name

In [11]:
def compute_prob(teams, score_method="538"):
    A_team_id, B_team_id = teams
    A_team_name = data.loc[A_team_id, '64']
    B_team_name = data.loc[B_team_id, '64']
    
    if score_method == "kenpom":
        A_em = data.loc[A_team_id, 'AdjustEM']
        A_t = data.loc[A_team_id, 'AdjustT']
        B_em = data.loc[B_team_id, 'AdjustEM']
        B_t = data.loc[B_team_id, 'AdjustT']
        sd = sd_params[0] + sd_params[1] * (((A_t + B_t) / 2) - avg_tempo) / avg_tempo
        d = norm(0, sd)
        diff = (A_em - B_em) * (A_t + B_t) / 200
        prob = d.cdf(diff)
    elif score_method == "538":
        A_538 = data.loc[A_team_id, 'team_rating']
        B_538 = data.loc[B_team_id, 'team_rating']
        diff = A_538 - B_538
        c = 30.464
        prob = 1 / (1 + 10**(-1 * diff * c / 400))
    else:
        raise Exception("Invalid score_method")
    
    return prob, A_team_id, A_team_name, B_team_id, B_team_name

In [12]:
def higher_prob(pyths, *args):
    prob, A_team_id, A_team_name, B_team_id, B_team_name = compute_prob(pyths)
    if prob >= 0.5:
        return A_team_id, A_team_name
    else:
        return B_team_id, B_team_name

In [13]:
def lower_prob(pyths, *args):
    prob, A_team_id, A_team_name, B_team_id, B_team_name = compute_prob(pyths)
    if prob >= 0.5:
        return B_team_id, B_team_name
    else:
        return A_team_id, A_team_name

In [14]:
def get_seeds(pyths, bracket):
    seed_A = bracket.loc[bracket["64"] == pyths[0][1], "Seed"].values[0]
    seed_B = bracket.loc[bracket["64"] == pyths[1][1], "Seed"].values[0]
    return seed_A, seed_B


def chalk(teams, bracket, *args):
    A_team_id, B_team_id = teams
    A_team_name = data.loc[A_team_id, '64']
    B_team_name = data.loc[B_team_id, '64']
    seed_A = data.loc[A_team_id, 'Seed']
    seed_B = data.loc[B_team_id, 'Seed']
    
    if seed_A <= seed_B:
        return A_team_id, A_team_name
    else:
        return B_team_id, B_team_name

In [15]:
def compute_winner_prob(pyths, winner_id):
    prob, A_team_id, A_team_name, B_team_id, B_team_name = compute_prob(pyths)
    
    if A_team_id == winner_id:
        return prob
    elif B_team_id == winner_id:
        return 1 - prob
    else:
        raise Exception("Hmm")

In [16]:
def compute_bracket_probability(bracket):
    rounds = ['64', '32', '16', '8', '4', '2', '1']
    probs = []
    for round_ in range(6):
        gap = 2**(round_ + 1)  # how far apart two teams could be for a given round
        for top_team in range(0, len(bracket), gap):
            pyths = get_teams(top_team, gap, bracket, rounds, round_, data)
            winner_id = get_teams(top_team, gap, bracket, rounds, round_ + 1, data)[0]
            prob = compute_winner_prob(pyths, winner_id)
            probs.append(prob)

    full_prob = np.prod(probs)  # compute the likelihood of the entire bracket
    return full_prob

In [17]:
bracket = create_specific_bracket(initial_bracket, data, espn_data, higher_prob)
prob = compute_bracket_probability(bracket)
prob

3.2464036792565433e-10

In [18]:
1/prob

3080331649.417701

In [19]:
bracket = create_specific_bracket(initial_bracket, data, espn_data, perfect_bracket)
prob = compute_bracket_probability(bracket)
prob

1.776828355052859e-15

In [20]:
1/prob

562800563800238.9

In [21]:
display(bracket.loc[bracket['8'] != ''][["Region", "Seed", "8", "4", "2", "1"]])

,Region,Seed,8,4,2,1
0,East,1,Connecticut,Connecticut,Connecticut,Connecticut
10,East,3,Illinois,,,
22,West,4,Alabama,Alabama,,
24,West,6,Clemson,,,
38,South,4,Duke,,,
41,South,11,North Carolina State,North Carolina State,,
48,Midwest,1,Purdue,Purdue,Purdue,
62,Midwest,2,Tennessee,,,


In [22]:
bracket = create_specific_bracket(initial_bracket, data, espn_data, lower_prob)
prob = compute_bracket_probability(bracket)
prob

8.620809739714172e-48

In [23]:
1/prob

1.1599838416491438e+47

In [24]:
bracket = create_specific_bracket(initial_bracket, data, espn_data, chalk)
prob = compute_bracket_probability(bracket)
prob

1.1574938244837707e-10

In [25]:
display(bracket.loc[bracket['8'] != ''][["Region", "Seed", "8", "4", "2", "1"]])

,Region,Seed,8,4,2,1
0,East,1,Connecticut,Connecticut,Connecticut,Connecticut
14,East,2,Iowa State,,,
16,West,1,North Carolina,North Carolina,,
30,West,2,Arizona,,,
32,South,1,Houston,Houston,Houston,
46,South,2,Marquette,,,
48,Midwest,1,Purdue,Purdue,,
62,Midwest,2,Tennessee,,,
